[Coursework fork](https://github.com/pelinsukk/remote-sensing-lc) · See README for data and reproducibility.

# Random forest classifier for landcover classification from remote sensing images

In this tasks, we will use a portion of the [SAT-6](https://csc.lsu.edu/~saikat/deepsat/) dataset from Louisiana State University and NASA Ames Research Center.

For this demonstration, we only use the test dataset, which consists of 81000 samples randomly distributed across 6 landcover classes.

In [ ]:
# Random forest classifier for landcover classification
# Author: Ankit Patnala, JSC, 2025; updated by Martin Schultz, April 2026
# Download necessary packages if right kernel is not present
# Not needed in colab
#!pip install pandas
#!pip install scikit-learn
#!pip install matplotlib

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier

## Obtaining the data

Download the three CSVs from the [B2SHARE record](https://b2share.eudat.eu/records/2p6ra-xfj86) into `data/`. Run from the repository root. Both models use identical balanced splits.

In [ ]:
import os
from sat6 import load_course_data, channel_columns

# Default: full coursework experiment. Smaller values are useful for smoke tests.
num_train = int(os.environ.get("SAT6_TRAIN_PER_CLASS", "1000"))
num_val = int(os.environ.get("SAT6_VAL_PER_CLASS", "100"))
num_test = int(os.environ.get("SAT6_TEST_PER_CLASS", "100"))
landcover_df, labels_df, annotations, (train_idx, val_idx, test_idx) = load_course_data(
    train=num_train, validation=num_val, test=num_test
)
np.random.seed(42)

In [ ]:
landcover_df.head()

&#x1F479;  _Task:_ Make sure you understand this data structure.

&#x1F479;  _Question:_ What would you need to do to extract only the green and the infrared channel from this data?

In [ ]:
labels = annotations[0].to_numpy()
print(labels)

The data structure that you see above is called _one-hot encoding_

&#x1F479; _Question:_ What is the advantage of this encoding compared to a simple class label like '0', '1', '2', '3', '4', '5', or text labels like 'building', 'barren_land', ...?

In [ ]:
column_names = labels_df.columns
labels_df.head()

# Plot random samples for illustration

&#x1F479; _Task:_ Make sure you understand the following code for creating balanced samples from the overall dataset and the reshaping of ther data that is needed for plotting.

In [ ]:
num_classes = len(column_names)
num_samples = min(8, num_train)  # number of rows to plot
sample_idx = []
for column in column_names:
    # find all indices of a given class
    class_idx = labels_df.loc[train_idx, column] == 1
    # randomly select num_samples from this index list - make sure to avoid duplicates
    sample_idx.append(np.random.choice(class_idx.index[class_idx], size=num_samples, replace=False).tolist())

fig, ax = plt.subplots(num_samples, num_classes, figsize=(20,20))
for i in range(num_samples):
    for j in range(num_classes):
        data_index = sample_idx[j][i]
        ax[i,j].set_title(f"{labels[j]}, {data_index}")
        ax[i,j].axis('off')
        data = np.reshape(landcover_df.loc[data_index].values,(-1,28,28,4))
        ax[i,j].imshow(data[0,...,:3])

plt.show()

### Shared training, validation and test split

One seeded permutation per class assigns rows to nonoverlapping splits. The validation split is used for depth and learning-curve comparisons; the final test split is held out.

In [ ]:
assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)
print(f"Train: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)}")

The original independently sampled splits could overlap. The shared splitter assigns each row to exactly one partition. Use `extend` to add individual row IDs and `append` to add a list as one element.

In [ ]:
train_X, train_y = landcover_df.loc[train_idx], labels_df.loc[train_idx]
val_X, val_y = landcover_df.loc[val_idx], labels_df.loc[val_idx]
test_X, test_y = landcover_df.loc[test_idx], labels_df.loc[test_idx]

&#x1F479; _Question:_ Why do you want to shuffle the samples in the train and test datasets?

In [ ]:
train_X

## Define and intialize random classifier from scikit-learn

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)

## Train the random forest

In [ ]:
rf.fit(train_X, train_y.to_numpy().argmax(axis=1))

## Check the prediction on the test set

In [ ]:
score_4ch = rf.score(test_X, test_y.to_numpy().argmax(axis=1))
print(f'Held-out four-band RF accuracy: {score_4ch:.6f}')

&#x1F479;  _Task:_ Add code to plot a couple of successes and failures (i.e., where predicted labels agree/disagree with the given labels in test_y)

In [ ]:
# Get predictions
pred_y = rf.predict(test_X)
true_int = np.argmax(test_y.values, axis=1)
pred_int = pred_y

correct_idx = np.where(true_int == pred_int)[0]
wrong_idx   = np.where(true_int != pred_int)[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top: Correct predictions  |  Bottom: Misclassifications', fontsize=13)

for i, idx in enumerate(correct_idx[:4]):
    img = np.reshape(test_X.iloc[idx].values, (28, 28, 4))
    axes[0, i].imshow(img[..., :3].astype(np.uint8))
    axes[0, i].set_title(f'True: {labels[true_int[idx]]}\nPred: {labels[pred_int[idx]]}')
    axes[0, i].axis('off')

for i, idx in enumerate(wrong_idx[:4]):
    img = np.reshape(test_X.iloc[idx].values, (28, 28, 4))
    axes[1, i].imshow(img[..., :3].astype(np.uint8))
    axes[1, i].set_title(f'True: {labels[true_int[idx]]}\nPred: {labels[pred_int[idx]]}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Task 2.2 — Per-class accuracy and classification examples

Here I evaluate the accuracy separately for each class using `classification_report`, and visualize a few correctly and incorrectly classified images.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# Per-class accuracy
print(classification_report(true_int, pred_int, labels=np.arange(6), target_names=labels, zero_division=0))

# Confusion matrix
cm = confusion_matrix(true_int, pred_int, labels=np.arange(6))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, cm[i, j], ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()
plt.show()

In [ ]:
# Plot correct and misclassified examples (same predictions from above)
n = 4
fig, axes = plt.subplots(2, n, figsize=(16, 8))
fig.suptitle('Top: Correct predictions  |  Bottom: Misclassifications', fontsize=13)

for i, idx in enumerate(correct_idx[:n]):
    img = np.reshape(test_X.iloc[idx].values, (28, 28, 4))
    axes[0, i].imshow(img[..., :3].astype(np.uint8))
    axes[0, i].set_title(f'True: {labels[true_int[idx]]}\nPred: {labels[pred_int[idx]]}')
    axes[0, i].axis('off')

for i, idx in enumerate(wrong_idx[:n]):
    img = np.reshape(test_X.iloc[idx].values, (28, 28, 4))
    axes[1, i].imshow(img[..., :3].astype(np.uint8))
    axes[1, i].set_title(f'True: {labels[true_int[idx]]}\nPred: {labels[pred_int[idx]]}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Task 2.3 — RGB only (no NIR)

The CSV is pixel-interleaved: each consecutive group of four values is R, G, B, NIR. RGB is band positions 0, 1 and 2 of each group; green and NIR would use `channel_columns(1, 3)`.

In [ ]:
rgb_cols = channel_columns(0, 1, 2)
rf_rgb = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
rf_rgb.fit(train_X.iloc[:, rgb_cols], train_y.to_numpy().argmax(axis=1))
score_rgb = rf_rgb.score(test_X.iloc[:, rgb_cols], test_y.to_numpy().argmax(axis=1))
print(f"Held-out RGB RF accuracy: {score_rgb:.6f}")

Compare the measured scores below rather than assuming that removing a band must lower accuracy.

## Task 2.4 — R, G, NIR only (no Blue)

Now I drop Blue instead of NIR to see which channel matters more.

In [ ]:
rgn_cols = channel_columns(0, 1, 3)
rf_rgn = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
rf_rgn.fit(train_X.iloc[:, rgn_cols], train_y.to_numpy().argmax(axis=1))
score_rgn = rf_rgn.score(test_X.iloc[:, rgn_cols], test_y.to_numpy().argmax(axis=1))
print(f"Held-out R+G+NIR RF accuracy: {score_rgn:.6f}")

The comparison is specific to this sample and model. It does not isolate a universally most informative band.

## Task 2.5 — Hyperparameter: max_depth

I chose `max_depth` because it directly controls how complex each tree is allowed to become.

**Default:** `None` — trees grow until all leaves are pure.
**New value:** `max_depth=10`

My expectation: limiting depth should act as regularization. Individual trees will be shallower and less likely to overfit. Accuracy might drop slightly but the model should generalize better.

In [ ]:
depths = [5, 10, 20, None]
results = {}

for d in depths:
    rf_d = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42, n_jobs=4)
    rf_d.fit(train_X, train_y.to_numpy().argmax(axis=1))
    score = rf_d.score(val_X, val_y.to_numpy().argmax(axis=1))
    results[str(d)] = score
    print(f'max_depth={str(d):6s}  ->  {score:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
xlabels = [str(d) for d in depths]
scores  = [results[k] for k in xlabels]
bars = ax.bar(xlabels, scores, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
ax.set_ylim(min(scores) - 0.05, 1.0)
ax.set_xlabel('max_depth')
ax.set_ylabel('Validation accuracy')
ax.set_title('Random Forest — effect of max_depth')
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{s:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f'\nWith max_depth=10 vs. unconstrained: {(results["None"] - results["10"])*100:.2f} pp difference')